# Notebook 2: Transforming the Data (Coding Variables for Analysis)

In Notebook 1 I cleaned the raw file. Here I turn the students' written answers and rating scales into variables I can count and compare.

1. Code the open-text answers (preferred method, faster method, willingness to pay) into categories
2. Flag the themes mentioned in the students' reasons
3. Create new columns with conditional logic (`np.where`)
4. Use NumPy for basic statistics
5. Use boolean masks to select groups
6. Group, aggregate, and build frequency tables, `pivot_table()` and `crosstab()`

**Why this matters in linguistic research:** an answer like "AI because it is fast" cannot be counted until it becomes a category
such as `AI`. This is a small version of what linguists call *coding* qualitative data.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/survey_notebook_clean.csv")
print(df.shape)

(32, 23)


## 1. Coding the preference question

In [2]:
# Let me look at some raw answers first, so I know what the function has to handle
for answer in df["preference_text"].head(8):
    print("-", answer)

- Human  translation 😋
- AI Translation because it is very fast
- Ai it’s easy and faster
- Human translation because it’s more accurate
- I prefer human translation because it’s not a literal translation, humans translate the actual meaning
- of course AI translation , it’s easier,for free, and useful
- Human translation is better because a person will take into account context and cultural background unlike AI which usually responds in a general manner
- nan


Rule I will use: **the first method named in the answer is the stated preference**
("Human, because..." is Human, "AI because it is fast" is AI).
The function reads the words one by one and stops at the first match.

In [3]:
def code_preference(answer):
    if pd.isna(answer):
        return "Not answered"
    # Turn the answer into a list of lower-case words, without commas and full stops
    text = answer.lower().replace(",", " ").replace(".", " ")
    words = text.split()
    for word in words:
        if word == "ai" or word == "ai's":
            return "AI"
        if word.startswith("human"):     # human, humans
            return "Human"
    return "Not answered"                # neither method named

df["preference"] = df["preference_text"].apply(code_preference)
print(df["preference"].value_counts())

preference
Human           17
AI              13
Not answered     2
Name: count, dtype: int64


In [4]:
# The function cannot understand every answer. Let me read the ones it could not code
# or that look odd, and decide by hand.
print(df[df["preference"] == "Not answered"][["participant_id", "preference_text"]])
print()
print(df[df["participant_id"] == "P32"]["preference_text"].values[0])

   participant_id                   preference_text
7             P08                               NaN
14            P15  no i don't, it's not as accurate

Depending on what I'm trying to translate for example if I want to translate something that's like poetic or like that has emotion in it I'll relay on human translation and if it's something more scientific and more like calculative I would depend on AI


- **P15** says "no i don't, it's not as accurate": this person does *not* prefer AI, so I code **Human**.
- **P32** says the choice depends on what is being translated, so I code **Depends**.
- **P08** left the question empty and stays "Not answered".

In [5]:
# loc[row_condition, column] = value changes specific cells
df.loc[df["participant_id"] == "P15", "preference"] = "Human"
df.loc[df["participant_id"] == "P32", "preference"] = "Depends"

print(df["preference"].value_counts())

preference
Human           17
AI              13
Not answered     1
Depends          1
Name: count, dtype: int64


## 2. Coding "which method is faster" and "willingness to pay"

In [6]:
# Same idea as before: first method mentioned. "google" counts as AI (Google Translate).
def code_faster(answer):
    if pd.isna(answer):
        return "Not answered"
    text = answer.lower().replace(",", " ").replace(".", " ")
    for word in text.split():
        if word == "ai" or word == "google":
            return "AI"
        if word.startswith("human"):
            return "Human"
    return "Unclear"          # for example "Idk"

df["faster_method"] = df["faster_text"].apply(code_faster)
print(df["faster_method"].value_counts())

faster_method
AI              23
Not answered     5
Unclear          2
Human            2
Name: count, dtype: int64


For willingness to pay, answers such as "Hell yeaaah", "Maybe", "If necessary" or "Because it's more accurate" cannot be sorted
reliably by keywords. With only 32 answers, the most honest method is to **read each answer and code it by hand**,
and to keep the list of IDs visible so the coding can be checked.

In [7]:
# I read all 32 answers and sorted the participant IDs into categories
yes_ids = ["P01", "P04", "P09", "P10", "P11", "P12", "P14", "P15", "P18", "P22", "P23", "P25", "P28"]
conditional_ids = ["P05", "P07", "P19", "P31"]      # "if it is a crucial situation", "in a professional environment"...
no_ids = ["P02", "P06", "P16", "P17", "P21", "P24", "P26", "P29", "P30", "P32"]
unclear_ids = ["P13", "P20"]                        # "Idk", "May be"

# Start with "Not answered" for everyone, then change the rows in each list
df["pay_more"] = "Not answered"
df.loc[df["participant_id"].isin(yes_ids), "pay_more"] = "Yes"
df.loc[df["participant_id"].isin(conditional_ids), "pay_more"] = "Conditional"
df.loc[df["participant_id"].isin(no_ids), "pay_more"] = "No"
df.loc[df["participant_id"].isin(unclear_ids), "pay_more"] = "Unclear"

print(df["pay_more"].value_counts())

pay_more
Yes             13
No              10
Conditional      4
Not answered     3
Unclear          2
Name: count, dtype: int64


## 3. Flagging themes in the reasons

In [8]:
# The students' reasons mention a few recurring ideas. I search for simple keywords.
# This is a rough first look. It is NOT a replacement for the thematic coding done in the thesis.
def mentions_any(answer, keywords):
    if pd.isna(answer):
        return False
    answer = answer.lower()
    for keyword in keywords:
        if keyword in answer:
            return True
    return False

speed_words = ["fast", "quick", "speed", "anytime", "whenever"]
ease_words = ["easy", "easier", "free", "accessible", "useful"]
accuracy_words = ["accura", "correct", "detail", "mistake"]
context_words = ["context", "cultur", "emotion", "meaning", "nuance", "tone", "human touch", "understand"]

df["theme_speed"] = df["preference_text"].apply(mentions_any, keywords=speed_words)
df["theme_ease_cost"] = df["preference_text"].apply(mentions_any, keywords=ease_words)
df["theme_accuracy"] = df["preference_text"].apply(mentions_any, keywords=accuracy_words)
df["theme_context_culture"] = df["preference_text"].apply(mentions_any, keywords=context_words)

theme_columns = ["theme_speed", "theme_ease_cost", "theme_accuracy", "theme_context_culture"]
print(df[theme_columns].sum())

theme_speed              10
theme_ease_cost           5
theme_accuracy            8
theme_context_culture    12
dtype: int64


*Limitation:* keyword searches can be wrong (a word can appear in a sentence with the opposite meaning), and one answer can contain several themes.
The counts are only used to describe the data, not to prove anything.

## 4. New columns with conditional logic

In [9]:
# Accuracy rating (1-10) into three bands. np.select handles more than two conditions.
conditions = [df["ai_accuracy_rating"] <= 5,
              df["ai_accuracy_rating"] <= 7]
labels = ["Low (1-5)", "Medium (6-7)"]
df["accuracy_band"] = np.select(conditions, labels, default="High (8-10)")

# Satisfaction into satisfied / not satisfied
df["satisfied"] = np.where(df["satisfaction"] >= 4, "Satisfied (4-5)", "Neutral or dissatisfied (1-3)")

# Has the student tried both AI and human translation?
df["used_both"] = np.where((df["used_ai"] == "Yes") & (df["used_human"] == "Yes"), "Yes", "No")

print(df["accuracy_band"].value_counts())
print(df["satisfied"].value_counts())
print(df["used_both"].value_counts())

accuracy_band
Medium (6-7)    16
High (8-10)     10
Low (1-5)        6
Name: count, dtype: int64
satisfied
Neutral or dissatisfied (1-3)    19
Satisfied (4-5)                  13
Name: count, dtype: int64
used_both
No     22
Yes    10
Name: count, dtype: int64


## 5. Descriptive statistics with NumPy

In [10]:
accuracy = df["ai_accuracy_rating"]       # a column of 32 numbers
satisfaction = df["satisfaction"]

print("ACCURACY (1-10)")
print("mean:", round(np.mean(accuracy), 2))
print("median:", np.median(accuracy))
print("standard deviation:", round(np.std(accuracy, ddof=1), 2))
print("min:", np.min(accuracy), " max:", np.max(accuracy))
print("25th and 75th percentiles:", np.percentile(accuracy, 25), np.percentile(accuracy, 75))
print()
print("SATISFACTION (1-5)")
print("mean:", round(np.mean(satisfaction), 2))
print("median:", np.median(satisfaction))
print("standard deviation:", round(np.std(satisfaction, ddof=1), 2))
print("min:", np.min(satisfaction), " max:", np.max(satisfaction))

ACCURACY (1-10)
mean: 6.75
median: 6.0
standard deviation: 1.57
min: 4  max: 10
25th and 75th percentiles: 6.0 8.0

SATISFACTION (1-5)
mean: 3.28
median: 3.0
standard deviation: 0.81
min: 1  max: 5


`ddof=1` makes NumPy compute the *sample* standard deviation (dividing by n - 1), which is the right one for a sample of students.
The same numbers can be obtained with `df["column"].describe()`, which I will use in Notebook 3.

In [11]:
# Looking for unusual ratings: more than 2 standard deviations from the mean
mean_accuracy = np.mean(accuracy)
sd_accuracy = np.std(accuracy, ddof=1)
unusual = df[np.abs(accuracy - mean_accuracy) > 2 * sd_accuracy]
print(unusual[["participant_id", "profession", "ai_accuracy_rating"]])

  participant_id profession  ai_accuracy_rating
5            P06     Writer                  10


One student gave 10/10. It is a valid answer inside the scale, so I **keep** it, but I remember it when I interpret the mean.

## 6. Boolean masks and conditional selection

In [12]:
# Students who rate AI accuracy 8 or more
high_accuracy = df[df["ai_accuracy_rating"] >= 8]
print("Rated AI accuracy 8+:", len(high_accuracy))
print("Of them, preferring AI:", (high_accuracy["preference"] == "AI").sum())
print()

# Students who prefer human translation AND would pay more
human_and_pay = df[(df["preference"] == "Human") & (df["pay_more"] == "Yes")]
print("Prefer human and would pay more:", len(human_and_pay))

# Students who prefer AI AND are not satisfied (1-3)
ai_not_satisfied = df[(df["preference"] == "AI") & (df["satisfaction"] <= 3)]
print("Prefer AI but satisfaction 3 or lower:", len(ai_not_satisfied))

Rated AI accuracy 8+: 10
Of them, preferring AI: 7

Prefer human and would pay more: 11
Prefer AI but satisfaction 3 or lower: 5


## 7. Grouping and aggregating

In [13]:
# Only keep students who clearly prefer AI or Human, for a clean comparison
df_two = df[df["preference"].isin(["AI", "Human"])]

# groupby() splits the table by preference, agg() calculates the statistics of each group
summary = df_two.groupby("preference")[["ai_accuracy_rating", "satisfaction"]].agg(["count", "mean", "std"])
print(summary.round(2))

           ai_accuracy_rating             satisfaction            
                        count  mean   std        count  mean   std
preference                                                        
AI                         13  7.77  1.42           13  3.69  0.63
Human                      17  6.06  1.14           17  3.18  0.64


In [14]:
# Frequency tables with value_counts(): counts, then percentages
counts = df["preference"].value_counts()
percentages = (counts / len(df) * 100).round(1)
print(pd.DataFrame({"n": counts, "percent": percentages}))

               n  percent
preference               
Human         17     53.1
AI            13     40.6
Not answered   1      3.1
Depends        1      3.1


In [15]:
# crosstab() counts how many students fall in each combination of two categories
print(pd.crosstab(df["preference"], df["pay_more"]))

pay_more      Conditional  No  Not answered  Unclear  Yes
preference                                               
AI                      2   6             1        2    2
Depends                 0   1             0        0    0
Human                   2   3             1        0   11
Not answered            0   0             1        0    0


In [16]:
# pivot_table(): average satisfaction for each preference and each usage frequency
pivot = df.pivot_table(values="satisfaction", index="preference", columns="ai_frequency", aggfunc="mean")
print(pivot.round(2))

ai_frequency  Always  Sometimes
preference                     
AI              4.25       3.44
Depends          NaN       2.00
Human           3.00       3.20
Not answered     NaN       1.00


Notice that some cells are empty (`NaN`) because nobody belongs to that combination, and some averages rest on only 1 student.
I will keep this in mind in Notebook 3.

## 8. Save the coded dataset

In [17]:
df.to_csv("../data/processed/survey_notebook_coded.csv", index=False)
print("Saved:", df.shape)

Saved: (32, 33)


Next: **Notebook 3** explores the research questions and makes the figures.